In [1]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("image-classification", model="prithivMLmods/Face-Mask-Detection")
pipe("https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/hub/parrots.png")

2025-08-20 11:56:44.095514: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-08-20 11:56:44.490840: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/home/jkl0909/miniconda3/envs/PCBAI/lib/python3.10/site-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/attr_value.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
/home/jkl0909/mi

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/372M [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/394 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Device set to use cuda:0


[{'label': 'Face_Mask Found', 'score': 0.9183529615402222},
 {'label': 'Face_Mask Not_Found', 'score': 0.08164703100919724}]

In [ ]:
import cv2
from transformers import pipeline
import numpy as np

# Initialize the pipeline
pipe = pipeline("image-classification", model="prithivMLmods/Face-Mask-Detection")

# Start webcam
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print("Cannot open camera")
    exit()

# Process webcam frames
print("Starting webcam, press 'q' to quit...")
while True:
    ret, frame = cap.read()
    if not ret:
        print("Failed to grab frame")
        break
    
    # Convert frame to RGB
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    
    # Run the pipeline
    results = pipe(rgb_frame)
    
    # Display results on the frame
    for result in results:
        label = result['label']
        score = result['score']
        text = f"{label}: {score:.2f}"
        cv2.putText(frame, text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    
    cv2.imshow("Mask Detection", frame)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
cap.release()
cv2.destroyAllWindows()

Device set to use cuda:0


Cannot open camera
Starting webcam, press 'q' to quit...
Failed to grab frame


[ WARN:0@166.934] global cap_v4l.cpp:914 open VIDEOIO(V4L2:/dev/video0): can't open camera by index
[ERROR:0@166.935] global obsensor_uvc_stream_channel.cpp:163 getStreamChannelGroup Camera index out of range


: 

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("andrewmvd/face-mask-detection")

print("Path to dataset files:", path)

Path to dataset files: /home/jkl0909/.cache/kagglehub/datasets/andrewmvd/face-mask-detection/versions/1


In [2]:
import os
import shutil
from pathlib import Path

# Explore the dataset structure
print("Dataset structure:")
for root, dirs, files in os.walk(path):
    level = root.replace(path, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = ' ' * 2 * (level + 1)
    for file in files[:5]:  # Show first 5 files only
        print(f"{subindent}{file}")
    if len(files) > 5:
        print(f"{subindent}... and {len(files) - 5} more files")
    if level > 3:  # Limit depth to avoid too much output
        break

Dataset structure:
1/
  images/
    maksssksksss158.png
    maksssksksss525.png
    maksssksksss27.png
    maksssksksss247.png
    maksssksksss328.png
    ... and 848 more files
  annotations/
    maksssksksss363.xml
    maksssksksss569.xml
    maksssksksss67.xml
    maksssksksss557.xml
    maksssksksss286.xml
    ... and 848 more files


In [3]:
import xml.etree.ElementTree as ET
import cv2
import yaml

# Create YOLO dataset structure
yolo_dataset_path = "/home/jkl0909/MaskDetection/yolo_dataset"
os.makedirs(f"{yolo_dataset_path}/images/train", exist_ok=True)
os.makedirs(f"{yolo_dataset_path}/images/val", exist_ok=True)
os.makedirs(f"{yolo_dataset_path}/labels/train", exist_ok=True)
os.makedirs(f"{yolo_dataset_path}/labels/val", exist_ok=True)

# Define class mapping
class_mapping = {
    'with_mask': 0,
    'without_mask': 1,
    'mask_weared_incorrect': 2
}

def convert_bbox_to_yolo(bbox, img_width, img_height):
    """Convert bounding box from XML format to YOLO format"""
    x_min, y_min, x_max, y_max = bbox
    
    # Calculate center coordinates and dimensions
    x_center = (x_min + x_max) / 2.0
    y_center = (y_min + y_max) / 2.0
    width = x_max - x_min
    height = y_max - y_min
    
    # Normalize coordinates
    x_center /= img_width
    y_center /= img_height
    width /= img_width
    height /= img_height
    
    return x_center, y_center, width, height

def parse_xml_annotation(xml_path):
    """Parse XML annotation file and extract bounding boxes"""
    tree = ET.parse(xml_path)
    root = tree.getroot()
    
    # Get image dimensions
    size = root.find('size')
    img_width = int(size.find('width').text)
    img_height = int(size.find('height').text)
    
    annotations = []
    for obj in root.findall('object'):
        class_name = obj.find('name').text
        if class_name in class_mapping:
            class_id = class_mapping[class_name]
            
            bbox = obj.find('bndbox')
            x_min = float(bbox.find('xmin').text)
            y_min = float(bbox.find('ymin').text)
            x_max = float(bbox.find('xmax').text)
            y_max = float(bbox.find('ymax').text)
            
            # Convert to YOLO format
            x_center, y_center, width, height = convert_bbox_to_yolo(
                [x_min, y_min, x_max, y_max], img_width, img_height
            )
            
            annotations.append(f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}")
    
    return annotations

print("Dataset preparation functions defined successfully!")

Dataset preparation functions defined successfully!


In [4]:
import random
from sklearn.model_selection import train_test_split

# Find all image and annotation files
annotations_dir = os.path.join(path, "annotations")
images_dir = os.path.join(path, "images")

if not os.path.exists(annotations_dir):
    print("Looking for annotations in the dataset...")
    # Find annotations directory
    for root, dirs, files in os.walk(path):
        if any(f.endswith('.xml') for f in files):
            annotations_dir = root
            break
    
    # Find images directory
    for root, dirs, files in os.walk(path):
        if any(f.lower().endswith(('.jpg', '.jpeg', '.png')) for f in files):
            images_dir = root
            break

print(f"Annotations directory: {annotations_dir}")
print(f"Images directory: {images_dir}")

# Get all annotation files
xml_files = [f for f in os.listdir(annotations_dir) if f.endswith('.xml')]
print(f"Found {len(xml_files)} annotation files")

# Split dataset into train and validation (80/20)
train_files, val_files = train_test_split(xml_files, test_size=0.2, random_state=42)

print(f"Training files: {len(train_files)}")
print(f"Validation files: {len(val_files)}")

def process_dataset(file_list, split):
    """Process a list of files for train or val split"""
    processed_count = 0
    
    for xml_file in file_list:
        # Get corresponding image file
        base_name = os.path.splitext(xml_file)[0]
        
        # Find image file with different extensions
        img_file = None
        for ext in ['.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG']:
            potential_img = os.path.join(images_dir, base_name + ext)
            if os.path.exists(potential_img):
                img_file = potential_img
                break
        
        if img_file is None:
            print(f"Warning: No image found for {xml_file}")
            continue
        
        # Parse XML annotation
        xml_path = os.path.join(annotations_dir, xml_file)
        try:
            annotations = parse_xml_annotation(xml_path)
            
            if annotations:  # Only process if there are valid annotations
                # Copy image to YOLO dataset
                img_dest = os.path.join(yolo_dataset_path, "images", split, base_name + os.path.splitext(img_file)[1])
                shutil.copy2(img_file, img_dest)
                
                # Save YOLO format labels
                label_dest = os.path.join(yolo_dataset_path, "labels", split, base_name + ".txt")
                with open(label_dest, 'w') as f:
                    f.write('\n'.join(annotations))
                
                processed_count += 1
                
                if processed_count % 100 == 0:
                    print(f"Processed {processed_count} files for {split}")
                    
        except Exception as e:
            print(f"Error processing {xml_file}: {e}")
    
    return processed_count

# Process training set
print("Processing training set...")
train_processed = process_dataset(train_files, "train")

# Process validation set
print("Processing validation set...")
val_processed = process_dataset(val_files, "val")

print(f"\nDataset conversion completed!")
print(f"Training samples: {train_processed}")
print(f"Validation samples: {val_processed}")
print(f"Total samples: {train_processed + val_processed}")

Annotations directory: /home/jkl0909/.cache/kagglehub/datasets/andrewmvd/face-mask-detection/versions/1/annotations
Images directory: /home/jkl0909/.cache/kagglehub/datasets/andrewmvd/face-mask-detection/versions/1/images
Found 853 annotation files
Training files: 682
Validation files: 171
Processing training set...
Processed 100 files for train
Processed 200 files for train
Processed 300 files for train
Processed 400 files for train
Processed 500 files for train
Processed 600 files for train
Processing validation set...
Processed 100 files for val

Dataset conversion completed!
Training samples: 682
Validation samples: 171
Total samples: 853


In [5]:
# Create YOLO dataset configuration file
dataset_config = {
    'path': yolo_dataset_path,
    'train': 'images/train',
    'val': 'images/val',
    'nc': 3,  # number of classes
    'names': ['with_mask', 'without_mask', 'mask_weared_incorrect']
}

# Save configuration to YAML file
config_path = os.path.join(yolo_dataset_path, "dataset.yaml")
with open(config_path, 'w') as f:
    yaml.dump(dataset_config, f, default_flow_style=False)

print(f"Dataset configuration saved to: {config_path}")
print("\nDataset configuration:")
print(yaml.dump(dataset_config, default_flow_style=False))

Dataset configuration saved to: /home/jkl0909/MaskDetection/yolo_dataset/dataset.yaml

Dataset configuration:
names:
- with_mask
- without_mask
- mask_weared_incorrect
nc: 3
path: /home/jkl0909/MaskDetection/yolo_dataset
train: images/train
val: images/val



In [7]:
# Install ultralytics if not already installed
import subprocess
import sys
import torch
from ultralytics.nn.tasks import DetectionModel
from torch.nn.modules.container import Sequential
from ultralytics.nn.modules import Conv

torch.serialization.add_safe_globals([DetectionModel, Sequential])

from ultralytics import YOLO
model = YOLO('yolov8n.pt')

# Note: YOLOv12 is not officially released yet, so we'll use YOLOv8n instead
# which has similar architecture and performance
print("Loading YOLOv8n model for training...")
model = YOLO('yolov8n.pt')  # Load a pretrained YOLOv8n model

print("Model loaded successfully!")
print(f"Model architecture: {model.model}")
print(f"Model parameters: {sum(p.numel() for p in model.model.parameters())}")

UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL ultralytics.nn.modules.Conv was not an allowed global by default. Please use `torch.serialization.add_safe_globals([ultralytics.nn.modules.Conv])` or the `torch.serialization.safe_globals([ultralytics.nn.modules.Conv])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch.load to learn more a

In [ ]:
# Train the model
print("Starting training...")
print(f"Dataset config: {config_path}")

# Training parameters
training_params = {
    'data': config_path,        # path to dataset YAML
    'epochs': 50,               # number of training epochs
    'imgsz': 640,               # image size
    'batch': 16,                # batch size (adjust based on GPU memory)
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',  # device to run on
    'workers': 4,               # number of workers for data loading
    'project': 'mask_detection_runs',  # project name
    'name': 'yolov8n_mask_detection',   # experiment name
    'save': True,               # save model checkpoints
    'plots': True,              # save plots
    'cache': False,             # cache images for faster training (use False for large datasets)
    'patience': 10,             # early stopping patience
    'save_period': 10,          # save checkpoint every N epochs
}

print("Training parameters:")
for key, value in training_params.items():
    print(f"  {key}: {value}")

# Start training
print("\n" + "="*50)
print("STARTING YOLO TRAINING")
print("="*50)

results = model.train(**training_params)

print("\n" + "="*50)
print("TRAINING COMPLETED!")
print("="*50)
print(f"Results: {results}")

In [ ]:
# Evaluate the trained model
print("Evaluating the trained model...")

# Load the best model from training
best_model_path = os.path.join('mask_detection_runs', 'yolov8n_mask_detection', 'weights', 'best.pt')
if os.path.exists(best_model_path):
    trained_model = YOLO(best_model_path)
    print(f"Loaded best model from: {best_model_path}")
else:
    trained_model = model
    print("Using current model for evaluation")

# Validate the model
validation_results = trained_model.val(data=config_path)

print("Validation Results:")
print(f"mAP50: {validation_results.box.map50:.4f}")
print(f"mAP50-95: {validation_results.box.map:.4f}")
print(f"Precision: {validation_results.box.mp:.4f}")
print(f"Recall: {validation_results.box.mr:.4f}")

# Test inference on a sample image
print("\nTesting inference on validation images...")
val_images_dir = os.path.join(yolo_dataset_path, "images", "val")
sample_images = [f for f in os.listdir(val_images_dir) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]

if sample_images:
    sample_image_path = os.path.join(val_images_dir, sample_images[0])
    results = trained_model.predict(sample_image_path, save=True, conf=0.5)
    print(f"Inference completed on sample image: {sample_images[0]}")
    print(f"Results saved to: runs/detect/predict/")
else:
    print("No validation images found for testing")

In [ ]:
# Save and export the trained model
print("Saving the trained model...")

# Save the model in different formats
if os.path.exists(best_model_path):
    model_to_export = YOLO(best_model_path)
else:
    model_to_export = trained_model

# Export to different formats
export_formats = ['onnx', 'torchscript']  # Can also add 'tflite', 'coreml' etc.

for format_type in export_formats:
    try:
        print(f"Exporting to {format_type.upper()}...")
        exported_path = model_to_export.export(format=format_type)
        print(f"✅ {format_type.upper()} export successful: {exported_path}")
    except Exception as e:
        print(f"❌ {format_type.upper()} export failed: {e}")

# Create a summary of the training
print("\n" + "="*60)
print("TRAINING SUMMARY")
print("="*60)
print(f"Dataset: Face Mask Detection")
print(f"Model: YOLOv8n")
print(f"Classes: {dataset_config['names']}")
print(f"Training samples: {train_processed}")
print(f"Validation samples: {val_processed}")
print(f"Best model saved at: {best_model_path}")
print(f"Validation mAP50: {validation_results.box.map50:.4f}")
print(f"Validation mAP50-95: {validation_results.box.map:.4f}")
print("="*60)

# Instructions for using the trained model
print("\nTo use the trained model:")
print(f"1. Load the model: model = YOLO('{best_model_path}')")
print("2. Make predictions: results = model.predict('image.jpg')")
print("3. Access results: results[0].boxes.data for bounding boxes")
print("\nTraining complete! 🎉")

In [2]:
conda install pytorch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 pytorch-cuda=12.1 -c pytorch -c nvidia
pip install -U ultralytics


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/1.0 MB ? eta -:--:--Downloading ultralytics-8.3.182-py3-none-any.whl (1.0 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 8.0 MB/s eta 0:00:00
  Attempting uninstall: ultralytics
    Found existing installation: ultralytics 8.0.239
    Uninstalling ultralytics-8.0.239:
      Successfully uninstalled ultralytics-8.0.239
  Attempting uninstall: ultralytics
    Found existing installation: ultralytics 8.0.239
    Uninstalling ultralytics-8.0.239:
      Successfully uninstalled ultralytics-8.0.239
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ultralyticsplus 0.1.0 requires ultralytics<8.1.0,>=8.0.225, but you have ultralytics 8.3.182 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all t